Para uma lente colocada na cintura de um feixe Gaussiano podemos dizer que:

$$
W_0' = \frac{W_0}{\sqrt{1 + (z_0/f)^2}}
$$

Nesse caso, podemos usar essa aproximação, porque nosso feixe está colimado. E pelas nossas medições o comprimento de Rayleigh desse feixe é de 2m aproximadamente, portanto estamos pela cintura dele.


E para o caso especial, em que a profundidade de foco ($2z_0$) é muito maior que a distância focal da lente podemos fazer a aproximaçáo:
$$
W_0' = \frac{\lambda}{\pi W_0}f
$$

Note que, é óbvio que isso também vale. A profundidade de foco do feixe é aproximadamente $4m$ e com toda certeza nossa lente terá uma distância focal na ordem de cm's.

O que temos agora é o raio da cintura do feixe sem nada. Queremos saber para qual lente teremos um raio que fará $z_r/L$ = 8. Sendo que,

$$
W_0 = \sqrt{\frac{\lambda z_0}{\pi}}
$$

Nosso primeiro objetivo é determinar a distância focal, tal que:
$$
\frac{z'_0}{L} = 8
$$
Ou seja sabemos que o $z_0' = 8L$ . Podemos, portanto, calcular o novo $W'_0$ e finalmente determinar o $f$.

In [25]:
import numpy as np
import matplotlib.pyplot as plt

In [26]:
### De nossas medições conseguimos determinar:
lam_pump = 325e-9 #m
crystal_L = 5e-3 #m
w_0 = 487.11e-6 #m


# definir funcoes para calcular:
calc_z0 = lambda lam, w0 : (np.pi *w0**2)/(lam**2)
calc_W0 = lambda lam,z0: np.sqrt((lam*z0)/np.pi)


z0 = calc_z0(lam_pump, w_0)

new_z0 = 8*crystal_L

new_w0 = calc_W0(lam_pump, new_z0)


In [27]:
print(new_z0*1e3, 'mm')
print(new_w0*1e6, 'µm')

40.0 mm
64.32750982580687 µm


In [28]:
define_f = lambda w0, w0new, lam: (w0*w0new*np.pi)/lam
lens_f = define_f(w_0,new_w0,lam_pump)

print(f'Focusing lens f = {lens_f*1e3:.2f}mm')

Focusing lens f = 302.89mm


Ou seja, para o nosso feixe gaussiano, com as especificações dadas acima nossa lente de focalização tem que ter $f \approx 300mm$, que é precisamente o que já está lá.

Temos uma restrição que são as lentes disponíveis no nosso laboratório, no nosso caso, a mais próxima ao valor desejada é a de $300mm$.

Para o feixe de coleta as coisas são mais complicadas.

Temos algumas garantias iniciais:
$$
\frac{z_{R_C}}{L} = 1 \rightarrow z_{R_C} = 5mm
$$
Já a fibra que temos é uma P1-630A-FC2, cujo Mode-Field Diameter é entre $3.6 \mu \text{m} $ e $5.3 \mu \text{m}$. Vamos considerar um valor intermediário de $4.45 \mu \text{m}$. A partir disso conseguimos determinar o diametro do feixe gaussiano contra-propagante:
$$
MFD = 2W_f → W_f = 2.225 \mu \text{m}
$$

Seguindo o caminho do contra-propagante, o feixe encontra uma lente asférica, assim:
$$
W_a = \frac{\lambda f_a}{\pi W_f}
$$

Podemos ainda supor $f$ a partir das especificações das lentes disponíveis. O feixe segue e encontra um sistema $4f$, que tem como função ajustar o tamanho desse feixe. Esse ponto conecta o $W'_C$ (associado a $z_{R_C}$ e a distância focal almejada) e W_a:

$$
\frac{W'_C}{W_a} = \frac{f_1}{f_2}
$$

Então, como $W'_C$:
$$
f = \frac{W_C * W'_C \pi}{\lambda}
$$
$$
f = W_a\frac{f_1}{f_2}W_C \frac{\pi}{\lambda}
$$
Por fim substituindo W_a:

$$
f = f_a \frac{f_1}{f_2}\frac{W_c}{W_f}
$$

Sendo que $W_C$ e $W_f$ são conhecidos e podemos fazer as escolhas mais razoáveis para o resto.

In [29]:
lam = 650e-9
z_R_C = crystal_L
# W_C:
W_C = calc_W0(lam, z_R_C)
#W_f:
MFD = 4.5e-6#m
W_f = MFD/2


def lens_f(fa, f1,f2, W_C, W_f):
  return fa*(f1/f2)*(W_C/W_f)



In [30]:
import ipywidgets as widgets
from IPython.display import display


lam = 650e-9          # wavelength [m]
crystal_L = 5e-3      # Rayleigh length target z_R,C [m]

# Fiber mode field diameter
MFD = 4.5e-6          # [m]
W_f = MFD / 2         # Gaussian mode radius [m]


def lens_f(fa, f1, f2, W_C, W_f):

    return fa * (f2 / f1) * (W_C / W_f)


def collimated_beam_after_asphere(lam, fa, W_f):
    """
    Radius of the approximately collimated Gaussian beam
    produced by the aspherical lens.

    fa must be in meters.
    """
    return lam * fa / (np.pi * W_f)


W_C = calc_W0(lam, crystal_L)

print(f"Target Rayleigh length z_R,C = {crystal_L*1e3:.2f} mm")
print(f"Target collection waist W_C = {W_C*1e6:.2f} µm")
print(f"Fiber mode radius W_f = {W_f*1e6:.2f} µm")
print(f"W_C / W_f = {W_C/W_f:.3f}")

Target Rayleigh length z_R,C = 5.00 mm
Target collection waist W_C = 32.16 µm
Fiber mode radius W_f = 2.25 µm
W_C / W_f = 14.295


In [31]:
# ============================================================
# INTERACTIVE LENS EXPLORER
# ============================================================

def interactive_lens_system(fa_mm, f1_mm, f2_mm):

    # Convert mm -> m where necessary
    fa = fa_mm * 1e-3

    # 4f magnification
    M = f2_mm / f1_mm

    # Beam radius after the aspherical lens
    W_asphere = collimated_beam_after_asphere(
        lam,
        fa,
        W_f
    )

    # Beam radius after the 4f telescope
    W_after_4f = W_asphere * M

    # Required collection-lens focal length
    # Since fa, f1, f2 are all entered in mm,
    # output is also in mm.
    f_C_mm = lens_f(
        fa_mm,
        f1_mm,
        f2_mm,
        W_C,
        W_f
    )

    print("==============================================")
    print("          COLLECTION SYSTEM DESIGN")
    print("==============================================")
    print()
    print(f"Wavelength                  = {lam*1e9:.1f} nm")
    print(f"Target Rayleigh length      = {crystal_L*1e3:.2f} mm")
    print()
    print(f"Fiber MFD                   = {MFD*1e6:.2f} µm")
    print(f"Fiber mode radius W_f       = {W_f*1e6:.2f} µm")
    print()
    print(f"Target collection waist W_C = {W_C*1e6:.2f} µm")
    print()
    print("----------------------------------------------")
    print(f"Aspherical lens f_a         = {fa_mm:.1f} mm")
    print(f"4f lens f1                  = {f1_mm:.1f} mm")
    print(f"4f lens f2                  = {f2_mm:.1f} mm")
    print()
    print(f"4f magnification f2/f1      = {M:.3f}")
    print()
    print(f"Beam radius after asphere   = {W_asphere*1e3:.3f} mm")
    print(f"Beam diameter after asphere = {2*W_asphere*1e3:.3f} mm")
    print()
    print(f"Beam radius after 4f        = {W_after_4f*1e3:.3f} mm")
    print(f"Beam diameter after 4f      = {2*W_after_4f*1e3:.3f} mm")
    print()
    print("----------------------------------------------")
    print(f"Required collection lens:")
    print()
    print(f"        f_C = {f_C_mm:.2f} mm")
    print()
    print("==============================================")


# ============================================================
# SLIDERS
# ============================================================

fa_widget = widgets.SelectionSlider(
    options=[8, 25],
    value=8,
    description='f_a [mm]:',
    continuous_update=False,
    style={'description_width': '100px'},
    layout=widgets.Layout(width='600px')
)

f1_widget = widgets.SelectionSlider(
    options  = [25.4, 35,75,  100,125,200,250, 300,400, 500, 750 ],
    value=100,
    description='f1 [mm]:',
    continuous_update=False,
    style={'description_width': '100px'},
    layout=widgets.Layout(width='600px')
)

f2_widget =  widgets.SelectionSlider(
    options  = [25.4, 35,75,  100,125,200,250, 300,400, 500, 750 ],
    value=100,
    description='f2 [mm]:',
    continuous_update=False,
    style={'description_width': '100px'},
    layout=widgets.Layout(width='600px')
)


# ============================================================
# DISPLAY
# ============================================================

widgets.interact(
    interactive_lens_system,
    fa_mm=fa_widget,
    f1_mm=f1_widget,
    f2_mm=f2_widget
);

interactive(children=(SelectionSlider(continuous_update=False, description='f_a [mm]:', layout=Layout(width='6…

Logo um exemplo razoável seria a escolha das lentes postas acima para fazer todo esse cálculo.